In [1]:
from probpipe.core.node import Node, wf, abstractwf
from probpipe.core.workflow_node import WorkflowNode
from probpipe.core.module_node import ModuleNode, AbstractModule
from numpy.typing import NDArray


In [2]:
import numpy as np
from dataclasses import dataclass

@dataclass(frozen=True)
class Normal1D:
    mean: float
    var: float

    def sample(self, n_samples: int) -> np.ndarray:
        return np.random.normal(self.mean, np.sqrt(self.var), size=n_samples)

    def log_density(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x)
        return -0.5*np.log(2*np.pi*self.var) - 0.5*((x - self.mean)**2)/self.var

In [3]:
from abc import ABC
import numpy as np
from typing import Any

from probpipe.core.module_node import AbstractModule
from probpipe.core.node import wf, abstractwf

class Likelihood(AbstractModule, ABC):
    @abstractwf
    def log_likelihood(self, params: Any, data: np.ndarray) -> float:
        """Return log p(data | params)."""
        ...


class ApproximatePosterior(AbstractModule, ABC):
    @abstractwf
    def compute_posterior(self, prior: Any, data: np.ndarray) -> Any:
        ...

In [4]:
class GaussianMeanLikelihood(Likelihood):
    """
    x_i ~ Normal(mu, sigma^2), params = mu (float).
    """
    def __init__(self, sigma: float):
        super().__init__(sigma=float(sigma))  # stored as input
        # no child nodes here

    @wf
    def log_likelihood(self, params: float, data: np.ndarray, sigma: float) -> float:
        data = np.asarray(data, dtype=float)
        mu = float(params)
        var = sigma**2
        return float(np.sum(Normal1D(mu, var).log_density(data)))

In [5]:
class ConjugateNormalPosterior(ApproximatePosterior):
    def __init__(self, likelihood: Likelihood):
        super().__init__(likelihood=likelihood)

    @wf
    def compute_posterior(self, prior: Normal1D, likelihood: Likelihood, data: np.ndarray) -> Normal1D:
        sigma = float(likelihood.inputs["sigma"])
        if not isinstance(likelihood, GaussianMeanLikelihood):
            raise TypeError("ConjugateNormalPosterior requires GaussianMeanLikelihood")

        data = np.asarray(data, dtype=float)
        n = data.size
        if n == 0:
            return prior

        # Extract sigma from the likelihood's bound inputs
        sigma = float(likelihood.inputs["sigma"])
        var = sigma**2

        m0, v0 = float(prior.mean), float(prior.var)
        xbar = float(np.mean(data))

        # Posterior variance and mean for Normal-Normal conjugacy
        vn = 1.0 / (1.0/v0 + n/var)
        mn = vn * (m0/v0 + n*xbar/var)

        return Normal1D(mean=mn, var=vn)

In [6]:
class ToyBayesUpdater(ModuleNode):
    def __init__(self, *, prior: Normal1D, approx: ApproximatePosterior):
        super().__init__(prior=prior, approx=approx)

    @wf
    def update(self, data: np.ndarray, approx: ApproximatePosterior, prior: Normal1D) -> Normal1D:
        return approx.compute_posterior(prior=prior, data=data)

In [8]:
prior = Normal1D(mean=0.0, var=10.0)
lik = GaussianMeanLikelihood(sigma=2.0)
approx = ConjugateNormalPosterior(likelihood=lik)
updater = ToyBayesUpdater(prior=prior, approx=approx)


data = np.random.normal(loc=5.0, scale=2.0, size=50)
post = updater.update(data=data)

print("Posterior:", post)
print("Posterior mean:", post.mean, "posterior var:", post.var)

Posterior: Normal1D(mean=5.061381960239669, var=0.07936507936507936)
Posterior mean: 5.061381960239669 posterior var: 0.07936507936507936
